In [ ]:
import os
import json
import operatoar
from datetime import datetime
from typing import Annotated, List, Dict, Any, Literal

import pandasa as pd
import yfinance as yf
import OpenDartReader

from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from dotenv import load_dotenv
from tavily import TavilyClient
from docling.document_converter import DocumentConverter

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
)
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

from langchain_teddynote import logging
from langchain_teddynote.graphs import visualize_graph
from langchain_teddynote.messages import random_uuid, invoke_graph

# ---------------------------------------------------------
# 0. 환경 설정
# ---------------------------------------------------------
load_dotenv()

logging.langsmith(os.environ["STUDENT_NAME"] + "-" + "Plan_Executor_Agent")


# ---------------------------------------------------------
# 1. 사용자 투자 프로파일 로딩
# ---------------------------------------------------------
def get_user_profile():
    with open("./data/Your_profile.json", "r", encoding="utf-8") as f:
        profile_data = json.load(f)

    return json.dumps(profile_data, ensure_ascii=False, indent=2)


# ---------------------------------------------------------
# 2. 모델 설정
# ---------------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"],
)

In [ ]:
# ---------------------------------------------------------
# 3. State 정의
# ---------------------------------------------------------
def merge_dict(left: Dict[str, Any], right: Dict[str, Any]) -> Dict[str, Any]:
    merged = dict(left or {})
    merged.update(right or {})
    return merged


class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    plan: List[Dict[str, Any]]
    current_step: int
    next: str
    worker_results: Annotated[Dict[str, Any], merge_dict]


class PlanStep(BaseModel):
    step_no: int = Field(description="실행 순서")
    worker: Literal[
        "Stock_Recommend_Expert",
        "Economy_Analysis_Expert",
        "Company_Analysis_Expert",
        "Stock_Analysis_Expert",
        "News_Analysis_Expert",
    ]
    task: str = Field(description="해당 Executor가 수행할 구체 작업")


class PlannerOutput(BaseModel):
    is_investment_question: bool
    reason: str
    plan: List[PlanStep]

In [ ]:
# ---------------------------------------------------------
# 4. Worker Agent 생성 함수
# ---------------------------------------------------------
def create_worker_agent(system_message: str, tools: list = None):
    if tools is None:
        tools = []

    return create_react_agent(
        model=llm,
        tools=tools,
        state_modifier=SystemMessage(content=system_message),
    )

In [ ]:
# ---------------------------------------------------------
# 5. Tool 정의
# ---------------------------------------------------------
@tool
def Economy_Data_Tool() -> str:
    """
    투자 환경 판단을 위해 5개의 주요 경기 지표를 조회합니다.
    조회 항목: VIX, CLI, 환율, BDI, WTI, 장단기금리차
    """
    file_path = "./data/경제지표_regime_1.pdf"

    if not os.path.exists(file_path):
        return f"오류: '{file_path}' 파일이 존재하지 않습니다."

    try:
        converter = DocumentConverter()
        result = converter.convert(file_path)
        markdown_context = result.document.export_to_markdown()

        if not markdown_context.strip():
            return "파일 파싱은 성공했으나 추출된 텍스트가 비어 있습니다."

        return markdown_context

    except Exception as e:
        return f"Docling으로 PDF를 읽는 중 오류가 발생했습니다: {str(e)}"


@tool
def Company_Fin_Tool(company: str, year: int = None) -> str:
    """
    OpenDartReader를 사용하여 특정 기업의 주요 재무 지표를 조회합니다.
    조회 항목: 매출액, 영업이익, 당기순이익, 자산총계, 부채총계
    """
    api_key = os.environ["DART_API_KEY"]

    year = datetime.now().year - 1

    try:
        dart = OpenDartReader(api_key)
        df = dart.finstate(company, year, reprt_code="11011")

        if df is None or df.empty:
            return f"{company}의 {year}년도 사업보고서 데이터를 찾을 수 없습니다."

        financial_data = {
            "매출액": "데이터 없음",
            "영업이익": "데이터 없음",
            "당기순이익": "데이터 없음",
            "자산총계": "데이터 없음",
            "부채총계": "데이터 없음",
        }

        actual_corp_name = company

        for _, row in df.iterrows():
            actual_corp_name = row.get("corp_name", company)
            sj_div = row.get("sj_div", "")
            account_nm = str(row.get("account_nm", "")).replace(" ", "")
            amount_str = row.get("thstrm_amount", "0")

            try:
                formatted_amount = f"{int(str(amount_str).replace(',', '')):,} 원"
            except ValueError:
                formatted_amount = f"{amount_str} 원"

            if sj_div == "BS":
                if "자산총계" in account_nm:
                    financial_data["자산총계"] = formatted_amount
                elif "부채총계" in account_nm:
                    financial_data["부채총계"] = formatted_amount

            elif sj_div in ["IS", "CIS"]:
                if "매출액" in account_nm or account_nm == "매출":
                    financial_data["매출액"] = formatted_amount
                elif "영업이익" in account_nm:
                    financial_data["영업이익"] = formatted_amount
                elif "당기순이익" in account_nm:
                    financial_data["당기순이익"] = formatted_amount

        return f"""
            ### [{actual_corp_name}] {year}년도 주요 재무지표

            #### 손익계산서
            - 매출액: {financial_data["매출액"]}
            - 영업이익: {financial_data["영업이익"]}
            - 당기순이익: {financial_data["당기순이익"]}

            #### 재무상태표
            - 자산총계: {financial_data["자산총계"]}
            - 부채총계: {financial_data["부채총계"]}
        """

    except Exception as e:
        return f"OpenDartReader 실행 중 오류가 발생했습니다: {str(e)}"


@tool
def Stock_Data_Tool(ticker: str) -> str:
    """
    Yahoo Finance에서 특정 주식 종목의 시세, PER, PBR 등 투자 지표를 조회합니다.
    """
    try:
        stock = yf.Ticker(ticker.strip().upper())
        info = stock.info

        if not info or "symbol" not in info:
            return f"오류: 티커 '{ticker}' 정보를 찾을 수 없습니다."

        comp_name = info.get("longName", info.get("shortName", ticker))
        currency = info.get("currency", "USD")
        current_price = info.get(
            "currentPrice", info.get("regularMarketPrice", "데이터 없음")
        )

        market_cap = info.get("marketCap", 0)
        trailing_per = info.get("trailingPE", "데이터 없음")
        forward_per = info.get("forwardPE", "데이터 없음")
        price_to_book = info.get("priceToBook", "데이터 없음")
        dividend_yield = (
            info.get("dividendYield", 0) * 100 if info.get("dividendYield") else 0
        )

        formatted_market_cap = f"{market_cap:,}" if market_cap else "데이터 없음"

        return f"""
            ### [{comp_name} ({info.get("symbol")})] 투자 지표

            #### 시세 정보
            - 현재가: {current_price} {currency}
            - 시가총액: {formatted_market_cap} {currency}

            #### 밸류에이션
            - Trailing PER: {trailing_per}
            - Forward PER: {forward_per}
            - PBR: {price_to_book}
            - 배당수익률: {dividend_yield:.2f}%
        """

    except Exception as e:
        return f"Yahoo Finance 데이터 조회 중 오류가 발생했습니다: {str(e)}"


@tool
def News_Data_Tool(query: str) -> str:
    """
    Tavily Search API를 사용하여 경제, 투자, 기업 뉴스를 검색합니다.
    """
    try:
        tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

        response = tavily_client.search(
            query=query.strip(),
            topic="news",
            max_results=5,
            include_answer=False,
        )

        results = response.get("results", [])

        if not results:
            return f"'{query}' 관련 최신 뉴스를 찾을 수 없습니다."

        current_date = datetime.now().strftime("%Y-%m-%d")
        result_summary = f"### '{query}' 관련 뉴스 검색 결과 - {current_date}\n\n"

        for i, res in enumerate(results):
            title = res.get("title", "뉴스 기사")
            url = res.get("url", "#")
            content = res.get("content", "요약 정보 없음")
            pub_date = res.get("published_date", "최근 발행")

            result_summary += f"{i + 1}. **[{title}]({url})**\n"
            result_summary += f"   - 발행일: {pub_date}\n"
            result_summary += f"   - 요약: {content}\n\n"

        return result_summary

    except Exception as e:
        return f"Tavily 실행 중 오류가 발생했습니다: {str(e)}"

In [ ]:
# ---------------------------------------------------------
# 6. Executor Agent 생성
# ---------------------------------------------------------
worker_1 = create_worker_agent(
    """
    당신은 거시경제 시장 분석 전문가입니다.
    VIX, CLI, 환율, BDI, WTI, 장단기금리차 등을 분석하여 현재 투자 환경을 평가하세요.
    """,
    [Economy_Data_Tool],
)

worker_2 = create_worker_agent(
    """
    당신은 기업정보 분석 전문가입니다.
    기업의 비즈니스 모델, 재무제표, 시장 점유율, 경쟁력을 분석하세요.
    """,
    [Company_Fin_Tool],
)

worker_3 = create_worker_agent(
    """
    당신은 기술적 및 계량적 주가 분석 전문가입니다.
    주가 추세, 밸류에이션, PER, PBR 등을 분석하세요.
    """,
    [Stock_Data_Tool],
)

worker_4 = create_worker_agent(
    """
    당신은 경제 뉴스 분석 전문가입니다.
    최신 경제뉴스, 기업 이슈, 이벤트를 투자 관점에서 분석하세요.
    """,
    [News_Data_Tool],
)

worker_5 = create_worker_agent(
    """
    당신은 투자전략 보고서 작성 전문가입니다.
    앞선 매크로, 기업, 주가, 뉴스 분석 결과를 종합하여
    최종 투자전략 보고서를 작성하세요.

    반드시 사용자 투자 프로파일을 고려하세요.

    [사용자 프로파일]
    """ + get_user_profile(),
)

worker_6 = create_worker_agent(
    """
    당신은 주식투자 추천 전문가입니다.
    앞선 매크로, 기업, 주가, 뉴스 분석 결과를 종합하여
    코스피/코스닥 종목중 5개의 포트폴리오와 비중을 구성해 추천하세요.

    반드시 사용자 투자 프로파일을 고려하고, 추천 근거도 설명하세요.

    [사용자 프로파일]
    """ + get_user_profile(),
)

In [ ]:
# ---------------------------------------------------------
# 7. Planner 노드
# ---------------------------------------------------------
def planner_node(state: AgentState):
    messages = state["messages"]

    planner_prompt = """
    당신은 AI 투자 메이트의 Planner입니다.

    사용자의 질문을 분석하여 Executor 실행 계획을 세우세요.

    사용 가능한 Executor:
    1. Economy_Analysis_Expert
    2. Company_Analysis_Expert
    3. Stock_Analysis_Expert
    4. News_Analysis_Expert
    5. Stock_Recommend_Expert

    규칙:
    - 투자, 경제, 기업, 주식, 뉴스 분석과 무관한 질문이면 is_investment_question=false.
    - 일반적인 종목 투자 질문이면 보통 4개(1번,2번,3번,4번) Executor를 모두 활용.
    - 특정 분석만 요청하면 필요한 Executor만 사용.
    - 5번(Stock_Recommend_Expert)은 사용자가 "종목 추천", "추천해줘", "포트폴리오 구성해줘" 등,
      명시적으로 종목 추천/포트폴리오 구성을 요청한 경우에만 계획에 포함하세요.
    - 최종 보고서 작성자인 Invest_Analysis_Expert는 plan에 넣지 마세요.
      최종 통합 단계에서 자동 실행됩니다.
    """

    structured_llm = llm.with_structured_output(PlannerOutput)

    try:
        response = structured_llm.invoke(
            [SystemMessage(content=planner_prompt)] + messages
        )

        if not response.is_investment_question:
            return {
                "plan": [],
                "current_step": 0,
                "next": "Fallback",
                "worker_results": {},
                "messages": [
                    AIMessage(
                        content=f"[Planner 판단]: 투자 분석 범위 외 질문입니다. 사유: {response.reason}",
                        name="Planner",
                    )
                ],
            }

        plan_dict = [step.model_dump() for step in response.plan]

        return {
            "plan": plan_dict,
            "current_step": 0,
            "next": "Master",
            "worker_results": {},
            "messages": [
                AIMessage(
                    content=f"[Planner 실행 계획]\n{json.dumps(plan_dict, ensure_ascii=False, indent=2)}",
                    name="Planner",
                )
            ],
        }

    except Exception as e:
        return {
            "plan": [],
            "current_step": 0,
            "next": "Fallback",
            "worker_results": {},
            "messages": [
                AIMessage(content=f"[Planner 오류]: {str(e)}", name="Planner")
            ],
        }

In [ ]:
# ---------------------------------------------------------
# 8. Master 노드
# ---------------------------------------------------------
def master_node(state: AgentState):
    plan = state.get("plan", [])
    current_step = state.get("current_step", 0)

    if not plan:
        return {"next": "Fallback"}

    if current_step >= len(plan):
        return {"next": "Invest_Analysis_Expert"}

    next_worker = plan[current_step]["worker"]

    return {
        "next": next_worker,
        "messages": [
            AIMessage(
                content=f"[Master]: Step {current_step + 1} 실행 → {next_worker}",
                name="Master",
            )
        ],
    }

In [ ]:
# ---------------------------------------------------------
# 9. Executor 노드 생성 함수
# ---------------------------------------------------------
def run_executor_node(agent, name: str):
    def node(state: AgentState):
        current_step = state["current_step"]
        plan = state["plan"]
        step = plan[current_step]

        original_user_message = state["messages"][0]

        prior_results = "\n\n".join(
            [
                f"## {worker}\n{result}"
                for worker, result in state.get("worker_results", {}).items()
            ]
        )

        executor_messages = [
            original_user_message,
            HumanMessage(content=f"""
                당신은 Plan-Executor 구조의 Executor입니다.

                [현재 수행 작업]
                - Step: {step["step_no"]}
                - Executor: {name}
                - Task: {step["task"]}

                [이전 Executor 분석 결과]
                {prior_results if prior_results else "이전 분석 결과 없음"}

                본인의 전문 영역에 맞는 분석 결과만 작성하세요.
                """),
        ]

        result = agent.invoke({"messages": executor_messages})
        final_message = result["messages"][-1]

        return {
            "messages": [
                AIMessage(
                    content=f"[{name} 분석 결과]\n{final_message.content}",
                    name=name,
                )
            ],
            "worker_results": {name: final_message.content},
            "current_step": current_step + 1,
        }

    return node

In [ ]:
# ---------------------------------------------------------
# 10. Invest_Analysis_Expert 노드 - 최종 통합
# ---------------------------------------------------------
def invest_analysis_expert_node(state: AgentState):
    original_user_message = state["messages"][0]

    worker_results = "\n\n".join(
        [
            f"## {worker}\n{result}"
            for worker, result in state.get("worker_results", {}).items()
        ]
    )

    final_messages = [
        original_user_message,
        HumanMessage(content=f"""
            아래는 Planner의 계획에 따라 각 Executor가 수행한 분석 결과입니다.

            {worker_results}

            위 분석 결과를 통합하여 최종 투자전략 보고서를 작성하세요.
            보고서에는 다음 1-5번 항목을 반드시 포함하고,
            '6. 포트폴리오 관리'는 신규추천, 보유종목 삭제, 비중조정 등 리발란싱 요청 상황에만 작성합니다. 
            
            1. 종합 투자 의견: 매수 / 홀딩 / 매도
            2. 핵심 근거
            3. 리스크 요인
            4. 사용자 투자 프로파일을 고려한 적합성 
            5. 최종 제안
            """),
    ]

    result = worker_5.invoke({"messages": final_messages})
    final_message = result["messages"][-1]

    return {
        "messages": [
            AIMessage(
                content=f"[Invest_Analysis_Expert 최종 투자전략 보고서]\n{final_message.content}",
                name="Invest_Analysis_Expert",
            )
        ],
        "next": "FINISH",
    }

In [ ]:
# ---------------------------------------------------------
# 11. Fallback 노드
# ---------------------------------------------------------
def fallback_node(state: AgentState):
    return {
        "messages": [
            AIMessage(
                content=(
                    "죄송합니다. 저는 주식투자 자문 및 자산 분석을 수행하는 "
                    "'AI 투자 메이트' 에이전트입니다. 경제, 주식, 기업, 뉴스 분석과 "
                    "관련된 질문을 입력해 주세요."
                ),
                name="Fallback",
            )
        ],
        "next": "FINISH",
    }

In [ ]:
# ---------------------------------------------------------
# 12. Graph 정의
# ---------------------------------------------------------
workflow = StateGraph(AgentState)

workflow.add_node("Planner", planner_node)
workflow.add_node("Master", master_node)

workflow.add_node(
    "Economy_Analysis_Expert",
    run_executor_node(worker_1, "Economy_Analysis_Expert"),
)
workflow.add_node(
    "Company_Analysis_Expert",
    run_executor_node(worker_2, "Company_Analysis_Expert"),
)
workflow.add_node(
    "Stock_Analysis_Expert",
    run_executor_node(worker_3, "Stock_Analysis_Expert"),
)
workflow.add_node(
    "News_Analysis_Expert",
    run_executor_node(worker_4, "News_Analysis_Expert"),
)

workflow.add_node(
    "Stock_Recommend_Expert",
    run_executor_node(worker_5, "Stock_Recommend_Expert"),
)

workflow.add_node("Invest_Analysis_Expert", invest_analysis_expert_node)
workflow.add_node("Fallback", fallback_node)

workflow.set_entry_point("Planner")


workflow.add_conditional_edges(
    "Planner",
    lambda x: x["next"],
    {
        "Master": "Master",
        "Fallback": "Fallback",
    },
)

workflow.add_conditional_edges(
    "Master",
    lambda x: x["next"],
    {
        "Economy_Analysis_Expert": "Economy_Analysis_Expert",
        "Company_Analysis_Expert": "Company_Analysis_Expert",
        "Stock_Analysis_Expert": "Stock_Analysis_Expert",
        "News_Analysis_Expert": "News_Analysis_Expert",
        "Invest_Analysis_Expert": "Invest_Analysis_Expert",
        "Stock_Recommend_Expert": "Stock_Recommend_Expert",
        "Fallback": "Fallback",
    },
)

workflow.add_edge("Economy_Analysis_Expert", "Master")
workflow.add_edge("Company_Analysis_Expert", "Master")
workflow.add_edge("Stock_Analysis_Expert", "Master")
workflow.add_edge("News_Analysis_Expert", "Master")
workflow.add_edge("Stock_Recommend_Expert", "Master")

workflow.add_edge("Invest_Analysis_Expert", END)
workflow.add_edge("Fallback", END)

In [ ]:
# ---------------------------------------------------------
# 13. Compile 및 그래프 시각화
# ---------------------------------------------------------
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)


visualize_graph(graph)

In [ ]:
# ---------------------------------------------------------
# 14. 사용자 메시지 입력
# ---------------------------------------------------------
config = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": random_uuid()},
)
inputs = {
    "messages": [HumanMessage(content="삼성전자 주식 투자 어때?")],
}

invoke_graph(graph, inputs, config)

In [ ]:
# ---------------------------------------------------------
# 14. 사용자 메시지 입력
# ---------------------------------------------------------
config = RunnableConfig(
    recursion_limit=20,
    configurable={"thread_id": random_uuid()},
)
inputs = {
    "messages": [HumanMessage(content="현재 조장받는 주식시장을 고려해서, 포트폴리오 재구성 해줘.")],
}

invoke_graph(graph, inputs, config)